# 02 - Feature Engineering & Feature Selection (Member 2)
**Project:** ML-based prediction of metabolic-syndrome risk from NHANES 2011-2014 wrist-accelerometer data.

This notebook uses **real project data only**. Nothing is simulated: when a required input is missing, the affected sections print exactly what is needed and are skipped, while independent sections still run.

| # | Section | Needs |
|---|---|---|
| 1-2 | Setup, data availability | - |
| 3-4 | Participant data, Set 0 (M2-01) | `clean_cohort_base.parquet` |
| 5-8 | Minute data, Set A, intensity, sedentary (M2-02..04) | `minute_streams_*` |
| 9-10 | PCA, K-Means (M2-05/06) | `minute_streams_*` |
| 11-12 | Selection, set comparison (M2-07/08) | Set 0 now; A/B need minute data |
| 13-14 | Export, validation | - |

**Leakage rules:** one stratified 80/20 split by `SEQN`; PCA, K-Means, imputers, scalers and selectors are fitted on training participants only; in cross-validation they are refitted inside every training fold; the target and target-defining clinical variables are never features.

## 1. Project setup and imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root = nearest parent folder containing src/features (independent of the working directory)
ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src" / "features").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project: a parent folder must contain src/features/")
sys.path.insert(0, str(ROOT))

from src.features.data_io import (
    TARGET, COHORT_REQUIRED, MINUTE_REQUIRED, COHORT_FILE_PATTERNS, MINUTE_FILE_PATTERNS,
    assert_no_target_leakage, candidate_data_dirs, check_data_availability, find_files,
    inventory_data_files, load_cohort, validate_cohort,
)
from src.features.minute_loader import load_minute_profiles
from src.features.build_features import (
    assemble_feature_sets, build_demographic_features, build_static_features,
    engineer_unsupervised_features, kmeans_k_scan, log_mims_matrix, pca_variance_table,
    pending_step_features,
)
from src.features.select_features import apply_feature_selection
from src.features.evaluate_sets import compare_feature_sets, stratified_split

RANDOM_STATE, TEST_SIZE, N_FOLDS = 42, 0.2, 5
REPORTS = ROOT / "reports"
FIG_DIR, TABLE_DIR = REPORTS / "figures", REPORTS / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(FIG_DIR / f"m2_{name}.png", dpi=150, bbox_inches="tight")

print("Project root:", ROOT)

## 2. Data availability checks
Searches the whole project (including a nested repo folder such as `mechine-learning-/`) for the two inputs and reports what is missing.

In [ ]:
avail = check_data_availability(ROOT)
for status in avail.values():
    print(status.describe())

HAS_COHORT = avail["cohort"].available
HAS_MINUTES = avail["minute"].available

dirs_with_cohort = [d for d in candidate_data_dirs(ROOT) if find_files(d, COHORT_FILE_PATTERNS)]
if len(dirs_with_cohort) > 1:
    print("\nNOTE: cohort file found in several folders; using the first:", [str(d) for d in dirs_with_cohort])

print("\nAll data files found in the project:")
display(inventory_data_files(ROOT))

if not HAS_MINUTES:
    print(f"\nBLOCKED INPUT - minute-level accelerometer data")
    print("  files   :", " or ".join(f"{p}.parquet/.csv" for p in MINUTE_FILE_PATTERNS),
          "(e.g. minute_streams_cycle_g.parquet, minute_streams_cycle_h.parquet from Member 1)")
    print("  columns :", MINUTE_REQUIRED)
    print("  blocked : sections 5-10, Set A / Set B, and the A/B parts of sections 11-13")
    print("  runnable: sections 3, 4, Set 0 parts of 11-13, 14")

## 3. Load and validate participant data
`validate_cohort` stops on missing columns, duplicate `SEQN`, a missing or non-binary target; missing values are reported. Survey weight (`WTSAF2YR`) and wear-QC columns are **not** features.

In [ ]:
if not HAS_COHORT:
    raise FileNotFoundError(avail["cohort"].describe())

COHORT_DIR = avail["cohort"].files[0].parent
PROCESSED_DIR = COHORT_DIR
cohort = load_cohort(COHORT_DIR)                       # validated, indexed by integer SEQN
report = validate_cohort(cohort.reset_index())
y = cohort[TARGET]
print(f"Loaded {avail['cohort'].files[0]}")
print(f"participants: {report['n_participants']} | target prevalence: {report['target_prevalence']:.3f}")
print("missing values in required columns:", report["missing_by_column"] or "none")
display(cohort.head())

# One reproducible stratified 80/20 split on the full cohort; every later analysis reuses it.
train_ids, test_ids = stratified_split(y, TEST_SIZE, RANDOM_STATE)
print(f"train = {len(train_ids)} ({y.loc[train_ids].mean():.3f} positive) | "
      f"test = {len(test_ids)} ({y.loc[test_ids].mean():.3f} positive)")

fig, ax = plt.subplots(figsize=(6, 3.2))
bands = pd.cut(cohort["RIDAGEYR"], [0, 30, 40, 50, 60, 70, 80, 120])
y.groupby(bands, observed=True).mean().plot.bar(ax=ax, color="#4C78A8")
ax.set(xlabel="age band", ylabel="metabolic syndrome prevalence", title="Target prevalence by age (all participants)")
plt.tight_layout(); save_fig(fig, "01_prevalence_by_age"); plt.show()

## 4. Demographic feature engineering - Feature Set 0 (M2-01)
Age, sex, race/ethnicity (one-hot), smoking (one-hot), survey cycle. Available in the handover: no education or income columns.

In [ ]:
set0_all = build_demographic_features(cohort)
assert_no_target_leakage(set0_all.columns)
print("Set 0:", set0_all.shape)
display(set0_all.head())
display(set0_all.describe().T[["mean", "std", "min", "max"]].round(3))

## 5. Minute-level accelerometer loading
Long format (`SEQN, PAXDAYM, PAXMTSM, PAXPREDM`) -> one typical-day 1440-minute MIMS profile per participant (complete 1440-minute days only, at least 3 per participant; non-wear and the -0.01 missing code excluded).

In [ ]:
minutes = days_per_id = None
if HAS_MINUTES:
    MINUTE_DIR = avail["minute"].files[0].parent
    minutes, days_per_id = load_minute_profiles(MINUTE_DIR, min_days=3)
    common = cohort.index.intersection(minutes.index)
    print(f"minute profiles: {minutes.shape} | participants in both tables: {len(common)} "
          f"(cohort {len(cohort)}, minute data {len(minutes)})")
    print("complete days per participant:"); display(days_per_id.describe().round(1).to_frame().T)
    # Matched participants for ALL feature sets; split assignments are inherited from section 3
    analysis_train = train_ids.intersection(common)
    analysis_test = test_ids.intersection(common)
else:
    print("SKIPPED: minute-level data not available (see section 2 for the exact files/columns needed).")
    analysis_train, analysis_test = train_ids, test_ids   # Set 0 analyses use the full cohort split
ids_all = analysis_train.union(analysis_test)
print(f"analysis participants: train {len(analysis_train)}, test {len(analysis_test)}")

## 6. Basic activity summaries - Feature Set A (M2-02)
Mean daily MIMS and wake/sleep wear minutes. Step-based summaries and activity counts are not in the handover -> pending (section 7).

In [ ]:
static = sets = None
if HAS_MINUTES:
    static = build_static_features(cohort, minutes)
    display(static["summaries"].describe().T[["mean", "std", "min", "max"]].round(2))
else:
    print("BLOCKED: needs minute-level data.")

## 7. Intensity and cadence features (M2-03)
`intensity_gradient` (log-log slope of time in MIMS bands) can be computed from MIMS. **Cadence/step features cannot**: the handover has no step columns, and steps are not derived from MIMS here.

In [ ]:
if HAS_MINUTES:
    display(static["intensity"].describe().T[["mean", "std", "min", "max"]].round(3))
pending = pending_step_features(minutes)
print("PENDING (need step-level data):")
display(pd.Series(pending, name="missing dependency").to_frame())

## 8. Sedentary and behavioural features (M2-04)
Sedentary fraction, bouts >= 30 min, breaks per sedentary hour, relative amplitude (M10/L5), day-to-day CV of MIMS. The sedentary MIMS cutoff (`SEDENTARY_MIMS = 10.5`) is an assumption to confirm.

In [ ]:
if HAS_MINUTES:
    beh = static["sedentary"].join(static["patterns"])
    display(beh.describe().T[["mean", "std", "min", "max"]].round(3))
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
    for ax, col in zip(axes, ["sedentary_fraction", "relative_amplitude"]):
        for lab, g in beh.loc[analysis_train].join(y).groupby(TARGET):
            ax.hist(g[col], bins=30, alpha=0.55, label=f"target={lab}", density=True)
        ax.set(title=col, xlabel=col); ax.legend()
    plt.tight_layout(); save_fig(fig, "08_behaviour_distributions"); plt.show()
else:
    print("BLOCKED: needs minute-level data.")

## 9. PCA of log10-MIMS profiles (M2-05)
Fitted on **training participants only**; the first 3 scores become features.

In [ ]:
if HAS_MINUTES:
    pca, var_table = pca_variance_table(minutes, analysis_train, n_components=10)
    display(var_table.round(4))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    axes[0].bar(var_table["component"], var_table["explained_variance_ratio"], color="#4C78A8")
    axes[0].plot(var_table["component"], var_table["cumulative"], "o-", color="#F58518")
    axes[0].set(xlabel="component", ylabel="explained variance", title="Scree plot (training)")
    scores = pca.transform(log_mims_matrix(minutes).loc[analysis_train])
    axes[1].scatter(scores[:, 0], scores[:, 1], c=y.loc[analysis_train], cmap="coolwarm", s=6, alpha=0.6)
    axes[1].set(xlabel="PC1", ylabel="PC2", title="PC1 vs PC2 (colour = target)")
    plt.tight_layout(); save_fig(fig, "09_pca"); plt.show()
else:
    print("BLOCKED: needs minute-level data.")

## 10. K-Means activity phenotypes (M2-06)
k = 4 (module default; confirm against the proposal). The scan below shows inertia/silhouette for context. All fitting uses training participants only.

In [ ]:
if HAS_MINUTES:
    scan = kmeans_k_scan(minutes, analysis_train, k_range=range(2, 9), random_state=RANDOM_STATE)
    display(scan.round(3))
    unsup = engineer_unsupervised_features(minutes, fit_index=analysis_train, random_state=RANDOM_STATE)
    sets = assemble_feature_sets(static, unsup)

    lm = log_mims_matrix(minutes).loc[analysis_train]
    lab = unsup.loc[analysis_train, "activity_phenotype_cluster"]
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    axes[0].plot(scan["k"], scan["inertia"], "o-"); axes[0].set(xlabel="k", title="Inertia")
    axes[1].plot(scan["k"], scan["silhouette"], "o-", color="#F58518"); axes[1].set(xlabel="k", title="Silhouette")
    for c, g in lm.groupby(lab):
        axes[2].plot(g.mean(axis=0).rolling(30, center=True, min_periods=1).mean().to_numpy(), label=f"cluster {c} (n={len(g)})")
    axes[2].set(xlabel="minute of day", ylabel="log10(MIMS+1)", title="Mean profile per phenotype"); axes[2].legend()
    plt.tight_layout(); save_fig(fig, "10_kmeans"); plt.show()
    display(y.loc[analysis_train].groupby(lab).agg(n="size", prevalence="mean").round(3))
else:
    print("BLOCKED: needs minute-level data.")

## 11. Feature selection (M2-07)
Correlation filter (|r| > 0.9) -> Mutual Information -> RFE -> L1 logistic regression, on **training participants only**. Runs for every feature set that can be built from the available data.

In [ ]:
set_frames = {"Set 0": set0_all.loc[ids_all]}
if HAS_MINUTES:
    set_frames["Set A"] = sets["setA"]
    set_frames["Set B"] = sets["setB"]

selection = {}
for name, X in set_frames.items():
    print(f"\n===== {name}: {X.shape[1]} candidate features, {len(analysis_train)} training participants =====")
    assert_no_target_leakage(X.columns)
    sel, rep = apply_feature_selection(X.loc[analysis_train], y.loc[analysis_train],
                                       return_report=True, random_state=RANDOM_STATE)
    selection[name] = (sel, rep)

stage_table = pd.DataFrame({
    n: {"candidates": set_frames[n].shape[1], "after |r|<=0.9": len(r["after_correlation"]),
        "after MI>0": len(r["after_mi"]), "after RFE": len(r["after_rfe"]), "after L1 (final)": len(s)}
    for n, (s, r) in selection.items()}).T
display(stage_table)
for n, (s, r) in selection.items():
    print(f"{n} final features ({len(s)}):", s, "| L1 fallback used:", r["l1_fallback"])

name_big = list(selection)[-1]
mi = selection[name_big][1]["mi_table"].head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(6, 4.2))
ax.barh(mi["Feature"], mi["MI_Score"], color="#4C78A8")
ax.set(xlabel="mutual information", title=f"Top MI scores - {name_big} (training only)")
plt.tight_layout(); save_fig(fig, "11_mutual_information"); plt.show()

## 12. Feature set comparison (M2-08)
Same participants and same stratified folds for every set. Imputer, scaler, **feature selection**, and (for Set B) PCA/K-Means are refitted inside each training fold. A plain logistic regression is used only as a yardstick for comparing sets; the held-out test participants are not touched.

In [ ]:
builders = {"Set 0": lambda fit_ids: set_frames["Set 0"]}
if HAS_MINUTES:
    builders["Set A"] = lambda fit_ids: set_frames["Set A"]
    builders["Set B"] = lambda fit_ids: assemble_feature_sets(
        static, engineer_unsupervised_features(minutes, fit_index=fit_ids, random_state=RANDOM_STATE))["setB"]
else:
    print("Set A and Set B comparison BLOCKED (no minute-level data); showing Set 0 only on the full cohort.")

comparison, details = compare_feature_sets(builders, y, analysis_train, n_splits=N_FOLDS, seed=RANDOM_STATE)
display(comparison.round(3))
display(pd.concat(details["folds"], names=["feature_set"]).round(3))

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.bar(comparison.index, comparison["roc_auc_mean"], yerr=comparison["roc_auc_std"], capsize=4, color="#4C78A8")
ax.set(ylabel=f"{N_FOLDS}-fold CV ROC-AUC (training only)", ylim=(0.5, 1.0), title="Feature set comparison")
plt.tight_layout(); save_fig(fig, "12_set_comparison"); plt.show()
print("Selection frequency across folds (Set 0):"); display(details["freq"]["Set 0"].to_frame())

## 13. Export results
Feature sets are saved with `SEQN`, the split and the target. `.gitignore` ignores `*.csv`, so parquet is the shareable format; `feature_set_b.csv` is written only when real Set B exists.

In [ ]:
def with_labels(X):
    out = X.join(cohort[[TARGET]], how="inner")
    out.insert(0, "split", np.where(out.index.isin(train_ids), "train",
                                    np.where(out.index.isin(test_ids), "test", "unassigned")))
    return out

exported = []
f0 = with_labels(set0_all); f0.to_parquet(PROCESSED_DIR / "feature_set_0.parquet"); exported.append("feature_set_0.parquet")
if HAS_MINUTES:
    fa, fb = with_labels(sets["setA"]), with_labels(sets["setB"])
    fa.to_parquet(PROCESSED_DIR / "feature_set_a.parquet")
    fb.to_parquet(PROCESSED_DIR / "feature_set_b.parquet")
    fb.to_csv(PROCESSED_DIR / "feature_set_b.csv")
    exported += ["feature_set_a.parquet", "feature_set_b.parquet", "feature_set_b.csv"]
else:
    print("NOT EXPORTED: feature_set_a/b (+ feature_set_b.csv) need minute-level data.")

pd.concat({n: pd.Series(s, name="feature") for n, (s, r) in selection.items()}, names=["feature_set", "rank"]) \
    .to_csv(TABLE_DIR / "m2_selected_features.csv")
comparison.to_csv(TABLE_DIR / "m2_feature_set_comparison.csv")
stage_table.to_csv(TABLE_DIR / "m2_selection_stages.csv")
print("exported to", PROCESSED_DIR, ":", exported)
print("tables ->", TABLE_DIR, "| figures ->", FIG_DIR)

## 14. Final validation and summary

In [ ]:
# Hard checks (these raise if violated)
for name, X in set_frames.items():
    assert X.index.is_unique, f"{name}: duplicate SEQN"
    assert_no_target_leakage(X.columns)
    assert TARGET not in X.columns
assert set(analysis_train).isdisjoint(analysis_test), "train/test overlap"
assert all(set(X.index) == set(ids_all) for X in set_frames.values()), "sets are not built on matched participants"
assert y.loc[ids_all].notna().all()
for n, (s, r) in selection.items():
    assert set(s) <= set(set_frames[n].columns)

status = pd.DataFrame([
    ("M2-01 Demographic features (Set 0)",        "WORKING",  "validated on real cohort"),
    ("M2-02 Basic activity summaries (Set A)",    "WORKING (MIMS/wear only)" if HAS_MINUTES else "BLOCKED", "step/AC summaries pending step data" if HAS_MINUTES else "needs minute data"),
    ("M2-03 Intensity & cadence",                 "PARTIAL (intensity gradient)" if HAS_MINUTES else "BLOCKED", "cadence features pending step data"),
    ("M2-04 Sedentary & activity patterns",       "WORKING" if HAS_MINUTES else "BLOCKED", "" if HAS_MINUTES else "needs minute data"),
    ("M2-05 PCA",                                 "WORKING" if HAS_MINUTES else "BLOCKED", "" if HAS_MINUTES else "needs minute data"),
    ("M2-06 K-Means phenotypes",                  "WORKING" if HAS_MINUTES else "BLOCKED", "" if HAS_MINUTES else "needs minute data"),
    ("M2-07 Feature selection (corr, MI, RFE, L1)", "WORKING", "verified on " + ", ".join(selection)),
    ("M2-08 Compare Sets 0/A/B + outputs",        "WORKING" if HAS_MINUTES else "PARTIAL (Set 0 only)", "" if HAS_MINUTES else "Set A/B need minute data"),
], columns=["task", "status", "note"])
display(status)
print("All validation checks passed." if HAS_MINUTES else
      "Validation passed for the sections that could run. Minute-level data is still required for the BLOCKED tasks.")